# Unit 14 — Diffusion sampling

Sampling starts with Gaussian noise. Each reverse update uses the denoiser prediction to move toward an action window. The policy then passes a queue prefix to the controller.

**Your result:** the code that turns noise into robot actions. Your reverse step equals the `diffusers` DDPM step at every timestep (including the final step with no noise and the jumps of a shortened schedule). Your sampler then produces the actions of a *real* policy object that you can run in the simulator.


**Sources.** [DDPM](https://arxiv.org/abs/2006.11239): equation 7 (posterior mean and variance), Algorithm 2 (sampling), equation 15 (the clean-sample estimate). [Diffusion Policy](https://diffusion-policy.cs.columbia.edu/) for which actions are executed. Pinned code (revision `e0d50211ef236143ae867228662b7dfaba554f02`): `DiffusionModel.conditional_sample`, `DiffusionModel.generate_actions` and `DiffusionPolicy.select_action`; the numerical step lives in `diffusers.DDPMScheduler.step`, which this unit uses **only as an oracle in checks**, never in your answers.

**Where this unit sits.** Units 11–13 are available as package versions (the forward process, the trained denoiser classes). The five functions of this unit are wired into a real policy wrapper through hooks; the wrapper never calls `scheduler.step`.


In [ ]:
import sys
from collections.abc import Callable, Sequence
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn

WORKSPACE = next(
    p
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "manifests/act_references.yaml").is_file()
)
sys.path.insert(0, str(WORKSPACE / "src"))
import ipywidgets as widgets
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from IPython.display import display

from vla_lab.sim.episode import open_libero_task, run_policy_episode
from vla_lab.teaching.diffusion import (
    CallbackDiffusionPolicy,
    DDPMSchedule,
    cosine_betas,
    executed_action_slice,
    load_prepared_diffusion,
    make_step_noises,
)

In [ ]:
# Supplied: a helper that the checks use
def close(actual: torch.Tensor, expected: torch.Tensor, message: str, atol: float = 1e-5) -> None:
    """Compare two tensors and say what failed."""
    assert actual.shape == expected.shape, (
        f"{message}: shape {tuple(actual.shape)}, expected {tuple(expected.shape)}"
    )
    worst = (actual.detach() - expected.detach()).abs().max().item()
    assert worst <= atol, f"{message}: largest difference {worst:.2e} is above {atol:.0e}"

## Background: running the process backwards

Training learned the noise predictor $\varepsilon_\theta(x_t, t)$. Sampling starts from pure noise $x_T\sim\mathcal{N}(0,I)$ and repeats one **reverse step** from the current step $t$ to an earlier step $t'$ until it reaches $x_0$:

1. Estimate the clean window, then **clip** it to $[-1, 1]$ (the pinned recipe has `clip_sample=True`; actions are scaled to $[-1, 1]$):
$$\hat{x}_0=\frac{x_t-\sqrt{1-\bar{\alpha}_t}\,\varepsilon_\theta(x_t,t)}{\sqrt{\bar{\alpha}_t}}$$
2. Take the posterior mean, with coefficients $c_0$ and $c_t$ from the *adjacent* or *skipped* step (Exercise 14.1):
$$\mu=c_0\,\hat{x}_0+c_t\,x_t$$
3. Add Gaussian noise with the posterior variance $\tilde{\beta}$, **except at the final step**: $x_{t'}=\mu+\sqrt{\tilde{\beta}}\,z$ with $z\sim\mathcal{N}(0,I)$.

Where:
* $x_t$ - the noisy action window at step $t$, shape $(B,H,A)$.
* $\bar{\alpha}_t$ - cumulative alpha at step $t$ (Unit 11).
* $\varepsilon_\theta(x_t,t)$ - the denoiser's noise prediction.
* $\hat{x}_0$ - the estimate of the clean window.
* $t'$ - the step the update moves to (`prev_t` in the code).
* $\mu$, $\tilde{\beta}$ - mean and variance of the Gaussian over $x_{t'}$.

**Shortened schedules.** The pinned policy trains with $T=100$ steps but samples with $n=20$ steps: $t=95, 90, \dots, 5, 0$. A jump from $t$ to $t'<t-1$ has no single $\beta$. Use the ratio of cumulative alphas, which is the product of the skipped alphas:

$$\alpha_{t\to t'}=\frac{\bar{\alpha}_t}{\bar{\alpha}_{t'}}, \qquad \beta_{t\to t'}=1-\alpha_{t\to t'}$$

**Which actions are executed.** The network predicts a window of 16 actions that starts *one frame before* the current observation (`n_obs_steps` $=2$). The policy executes indices $n_o-1,\dots,n_o+n_a-2$ $=1,\dots,8$, where $n_o$ is `n_obs_steps` and $n_a$ is `n_action_steps`, then samples again.
### Worked example: one skipped reverse update

Let $\bar{\alpha}_t=0.5$ and $\bar{\alpha}_{t'}=0.8$. The jump keeps $\alpha_{t\to t'}=0.5/0.8=0.625$ and uses $\beta_{t\to t'}=0.375$. The posterior weights are about $c_0=0.671$, $c_t=0.316$, and variance $\tilde{\beta}=0.150$. Thus the mean is $0.671\,\hat{x}_0+0.316\,x_t$; add noise with standard deviation $\sqrt{0.150}\approx0.387$ except at the final step. The symbols are scalar schedule values; they multiply every value in $(B,H,A)$.

### Why the posterior variance has a ratio

At step $t$, the forward marginal has noise variance $1-\bar{\alpha}_t$. At the earlier step $t'$, its noise variance is $1-\bar{\alpha}_{t'}$. For a jump, the posterior variance is the jump variance multiplied by their ratio:

$$\tilde{\beta}=\frac{1-\bar{\alpha}_{t'}}{1-\bar{\alpha}_t}\,\beta_{t\to t'}$$

This conditional factor accounts for how much uncertainty was already present at $t'$ relative to the total at $t$; it is not another beta schedule.

In the numerical example above, the fraction is $(1-0.8)/(1-0.5)=0.4$, so the posterior variance is $0.4\cdot0.375=0.15$. At the final clean step, $\bar{\alpha}_{t'}=1$, so the numerator is zero: the final step adds no noise.


## Exercise 14.1 — Posterior coefficients for adjacent and skipped steps

**Purpose.** Derive the weights of the reverse step. This is where a shortened schedule differs from the textbook formula: the one-step $\beta_t$ no longer applies.

**Contract (shapes, units, masks).**

- `posterior_coefficients(abar_t, abar_prev)` takes the cumulative alphas at the current step and at the step we jump to (python floats or 0-d tensors). `abar_prev = 1` means "the clean sample" (used for the final step).
- Compute `alpha_t` $=\alpha_{t\to t'}=\bar{\alpha}_t/\bar{\alpha}_{t'}$ and `beta_t` $=\beta_{t\to t'}=1-\alpha_{t\to t'}$.
- Return three 0-d `float32` tensors `(coef_x0, coef_xt, variance)`:

$$c_0=\frac{\sqrt{\bar{\alpha}_{t'}}\,\beta_{t\to t'}}{1-\bar{\alpha}_t}, \qquad c_t=\frac{\sqrt{\alpha_{t\to t'}}\left(1-\bar{\alpha}_{t'}\right)}{1-\bar{\alpha}_t}, \qquad \tilde{\beta}=\max\!\left(\frac{1-\bar{\alpha}_{t'}}{1-\bar{\alpha}_t}\,\beta_{t\to t'},\ 10^{-20}\right)$$

  Where $\bar{\alpha}_t$ is `abar_t`, $\bar{\alpha}_{t'}$ is `abar_prev`, and $c_0$, $c_t$, $\tilde{\beta}$ are `coef_x0`, `coef_xt`, `variance`.
- For `abar_prev = 1` the result is `coef_x0 = 1`, `coef_xt = 0` (the mean is the clean estimate).

**Method and source.** [DDPM](https://arxiv.org/abs/2006.11239), equation 7, gives the posterior over the earlier step:

$$q(x_{t'}\mid x_t,x_0)=\mathcal{N}\!\left(\tilde{\mu},\ \tilde{\beta}I\right), \qquad \tilde{\mu}=c_0\,x_0+c_t\,x_t$$

Where $c_0$, $c_t$, $\tilde{\beta}$ are the contract's three values. Skipped steps: apply the same equation to the coarser chain whose one-step alpha is $\bar{\alpha}_t/\bar{\alpha}_{t'}$. Why this works: the marginals $q(x_t\mid x_0)$ are the same Gaussians whatever steps we skip, so the chain over the chosen steps has per-jump alpha $\bar{\alpha}_t/\bar{\alpha}_{t'}$.

**Pointers.** Write the ratio and `beta_t` first. Then the three formulas in the order of the contract. Use `torch.as_tensor(..., dtype=torch.float32)` so floats and tensors both work. Derive for yourself why `alpha_t * abar_prev = abar_t`.

<details><summary>Detailed hint</summary>

```python
abar_t = torch.as_tensor(abar_t, dtype=torch.float32)
abar_prev = torch.as_tensor(abar_prev, dtype=torch.float32)
alpha_t = abar_t / abar_prev
beta_t = 1.0 - alpha_t
coef_x0 = abar_prev.sqrt() * beta_t / (1.0 - abar_t)
coef_xt = alpha_t.sqrt() * (1.0 - abar_prev) / (1.0 - abar_t)
variance = ((1.0 - abar_prev) / (1.0 - abar_t) * beta_t).clamp_min(1e-20)
return coef_x0, coef_xt, variance
```

</details>

More behaviour checks for this exercise: `check_posterior_coefficients` in [lesson14_checks.py](lesson14_checks.py).

In [ ]:
# Implement: Exercise 14.1
def posterior_coefficients(
    abar_t: torch.Tensor | float, abar_prev: torch.Tensor | float
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # Return (coef_x0, coef_xt, variance) for a jump from step t to an earlier step with cumulative alpha abar_prev.
    raise NotImplementedError("Complete Exercise 14.1: posterior_coefficients")

In [ ]:
# Check: Exercise 14.1
scheduler = DDPMScheduler(num_train_timesteps=100, beta_schedule="squaredcos_cap_v2")
abar, betas = scheduler.alphas_cumprod, scheduler.betas
for t, prev in ((50, 49), (50, 45), (99, 94), (5, 0)):
    c0, ct, var = posterior_coefficients(abar[t], abar[prev])
    assert c0.shape == () and ct.shape == () and var.shape == (), "Return three scalar tensors."
    product = torch.prod(1 - betas[prev + 1 : t + 1])  # the skipped alphas multiplied
    close(abar[t] / abar[prev], product, f"alpha_t for the jump {t} -> {prev}", atol=1e-6)
    beta = 1 - product
    close(c0, abar[prev].sqrt() * beta / (1 - abar[t]), f"coef_x0 for {t} -> {prev}", atol=1e-6)
    close(
        ct,
        product.sqrt() * (1 - abar[prev]) / (1 - abar[t]),
        f"coef_xt for {t} -> {prev}",
        atol=1e-6,
    )
    close(var, (1 - abar[prev]) / (1 - abar[t]) * beta, f"variance for {t} -> {prev}", atol=1e-7)
c0, ct, var = posterior_coefficients(abar[0], 1.0)
close(c0, torch.tensor(1.0), "final step: coef_x0 is 1", atol=1e-6)
close(ct, torch.tensor(0.0), "final step: coef_xt is 0", atol=1e-6)
assert 0 < float(var) <= 1e-19, "Clamp the variance to at least 1e-20 (it is 0 at the final step)."
c0b, _, _ = posterior_coefficients(float(abar[50]), float(abar[40]))
close(c0b, posterior_coefficients(abar[50], abar[40])[0], "python floats must work too", atol=1e-7)
print("Exercise 14.1 passed")

# More behaviour checks: lesson14_checks.py, next to this notebook.
from lesson14_checks import check_posterior_coefficients

check_posterior_coefficients(posterior_coefficients)

## Exercise 14.2 — One reverse step (with the final step)

**Purpose.** Write the single update that the sampler repeats. The final step is special: there is no noise and the result is the clipped clean estimate.

**Contract (shapes, units, masks).**

- `ddpm_reverse_step(x_t, epsilon, t, prev_t, alpha_bar, noise=None, clip_range=1.0)`.
- `x_t`, `epsilon`, optional `noise`: `(B, H, A)`. `t`, `prev_t`: python ints; `prev_t < 0` means "the final step" (after it there is no earlier step). `alpha_bar`: `(T,)` cumulative alphas.
- `x0_hat` $=\hat{x}_0=\left(x_t-\sqrt{1-\bar{\alpha}_t}\,\varepsilon\right)/\sqrt{\bar{\alpha}_t}$, then clamp to `[-clip_range, clip_range]` if `clip_range` is not `None`.
- `abar_prev = alpha_bar[prev_t]`, or exactly `1` when `prev_t < 0`. Get coefficients from your Exercise 14.1 function. `mean = coef_x0 * x0_hat + coef_xt * x_t`.
- If `prev_t >= 0`: return `mean + sqrt(variance) * noise`; raise `ValueError` if `noise` is `None`. If `prev_t < 0`: return `mean` and **never** use `noise` (it may be `None` or any value).
- Use only the passed tensors. **Do not call a scheduler.**

**Method and source.** DDPM Algorithm 2, line 4: $x_{t-1}=\tilde{\mu}(x_t,\hat{x}_0)+\sigma_t z$, where $\sigma_t=\sqrt{\tilde{\beta}}$ and $z\sim\mathcal{N}(0,I)$ is `noise`, with $z=0$ at the last step; $\hat{x}_0$ from equation 15 and clipping as in the reference implementation. This is what `DDPMScheduler.step` computes with `prediction_type='epsilon'`.

**Pointers.** Estimate `x0_hat`, clip, fetch `abar_prev` (watch the final-step convention), call `posterior_coefficients`, combine. Draw no random numbers inside this function: the caller passes the noise.

<details><summary>Detailed hint</summary>

```python
abar_t = alpha_bar[t].to(x_t.dtype)
abar_prev = alpha_bar[prev_t].to(x_t.dtype) if prev_t >= 0 else torch.ones((), dtype=x_t.dtype)
x0_hat = (x_t - (1.0 - abar_t).sqrt() * epsilon) / abar_t.sqrt()
if clip_range is not None:
    x0_hat = x0_hat.clamp(-clip_range, clip_range)
coef_x0, coef_xt, variance = posterior_coefficients(abar_t, abar_prev)
mean = coef_x0.to(x_t.dtype) * x0_hat + coef_xt.to(x_t.dtype) * x_t
if prev_t < 0:
    return mean
if noise is None:
    raise ValueError("noise is required for every step except the final one")
return mean + variance.sqrt().to(x_t.dtype) * noise
```

</details>

More behaviour checks for this exercise: `check_ddpm_reverse_step` in [lesson14_checks.py](lesson14_checks.py).

In [ ]:
# Implement: Exercise 14.2
def ddpm_reverse_step(
    x_t: torch.Tensor,
    epsilon: torch.Tensor,
    t: int,
    prev_t: int,
    alpha_bar: torch.Tensor,
    noise: torch.Tensor | None = None,
    clip_range: float | None = 1.0,
) -> torch.Tensor:
    # x0_hat (clipped) -> posterior mean -> add sqrt(variance) * noise unless this is the final step.
    raise NotImplementedError("Complete Exercise 14.2: ddpm_reverse_step")

In [ ]:
# Check: Exercise 14.2
for steps in (5, 20, 100):
    scheduler = DDPMScheduler(
        num_train_timesteps=100,
        beta_schedule="squaredcos_cap_v2",
        clip_sample=True,
        clip_sample_range=1.0,
    )
    scheduler.set_timesteps(steps)
    times = scheduler.timesteps.tolist()
    abar = scheduler.alphas_cumprod
    g = torch.Generator().manual_seed(0)
    for index, t in enumerate(times):
        prev = times[index + 1] if index + 1 < len(times) else -1
        x = torch.randn(2, 16, 7, generator=g) * 1.5
        eps = torch.randn(2, 16, 7, generator=g)
        noise = torch.randn(2, 16, 7, generator=torch.Generator().manual_seed(100 + index))
        expected = scheduler.step(
            eps, t, x, generator=torch.Generator().manual_seed(100 + index)
        ).prev_sample  # oracle only
        mine = ddpm_reverse_step(x, eps, t, prev, abar, noise if prev >= 0 else None, 1.0)
        close(
            mine,
            expected,
            f"{steps}-step schedule, t={t} -> prev={prev} differs from diffusers",
            atol=1e-5,
        )
        if prev < 0:
            x0 = ((x - (1 - abar[t]).sqrt() * eps) / abar[t].sqrt()).clamp(-1, 1)
            close(mine, x0, "the final step returns the clipped clean estimate", atol=1e-5)
            close(
                ddpm_reverse_step(x, eps, t, prev, abar, torch.full_like(x, 99.0), 1.0),
                mine,
                "the final step must ignore any noise",
                atol=0,
            )
abar = DDPMSchedule.cosine(100).alpha_bar
try:
    ddpm_reverse_step(torch.zeros(1, 2, 2), torch.zeros(1, 2, 2), 10, 5, abar, None)
except ValueError:
    pass
else:
    raise AssertionError("A non-final step without noise must raise ValueError.")
unclipped = ddpm_reverse_step(
    torch.full((1, 2, 2), 3.0), torch.zeros(1, 2, 2), 0, -1, abar, None, clip_range=None
)
clipped = ddpm_reverse_step(
    torch.full((1, 2, 2), 3.0), torch.zeros(1, 2, 2), 0, -1, abar, None, clip_range=1.0
)
assert float(unclipped.max()) > 1.5 and float(clipped.max()) <= 1.0 + 1e-6, (
    "clip_range=None must not clamp; 1.0 must."
)
print("Exercise 14.2 passed")

# More behaviour checks: lesson14_checks.py, next to this notebook.
from lesson14_checks import check_ddpm_reverse_step

check_ddpm_reverse_step(ddpm_reverse_step)

## Exercise 14.3 — Timesteps of a shortened schedule

**Purpose.** Choose which of the 100 training steps the sampler visits. Fewer steps are faster; the choice must keep the last step at 0.

**Contract (shapes, units, masks).**

- `inference_timesteps(num_train_timesteps, num_inference_steps)` returns a Python `list` of ints in **descending** order.
- With $r=\lfloor T/n\rfloor$ the list has length $n$:

$$\left[(n-1)\,r,\ \dots,\ 2r,\ r,\ 0\right]$$

  Where $T$ is `num_train_timesteps`, $n$ is `num_inference_steps`, and $r$ is the stride between visited steps. For $T=100$, $n=20$ that is $[95, 90, \dots, 5, 0]$.
- The last entry is always `0`. Raise `ValueError` if `n < 1` or `n > T`.

**Method and source.** Evenly spaced ('leading') timesteps as in `diffusers` `DDPMScheduler.set_timesteps`; the pinned policy samples 20 of its 100 steps.

**Pointers.** Make the ascending multiples of the ratio, then reverse them. Validate the inputs first.

<details><summary>Detailed hint</summary>

```python
if not 1 <= num_inference_steps <= num_train_timesteps:
    raise ValueError("num_inference_steps must be between 1 and the training step count")
ratio = num_train_timesteps // num_inference_steps
return [index * ratio for index in reversed(range(num_inference_steps))]
```

</details>

More behaviour checks for this exercise: `check_inference_timesteps` in [lesson14_checks.py](lesson14_checks.py).

In [ ]:
# Implement: Exercise 14.3
def inference_timesteps(num_train_timesteps: int, num_inference_steps: int) -> list[int]:
    # Descending evenly spaced training timesteps ending at 0.
    raise NotImplementedError("Complete Exercise 14.3: inference_timesteps")

In [ ]:
# Check: Exercise 14.3
for steps in (1, 5, 10, 20, 50, 100):
    scheduler = DDPMScheduler(num_train_timesteps=100, beta_schedule="squaredcos_cap_v2")
    scheduler.set_timesteps(steps)
    mine = inference_timesteps(100, steps)
    assert isinstance(mine, list) and mine == scheduler.timesteps.tolist(), (
        f"{steps} steps: expected {scheduler.timesteps.tolist()[:4]}…, got {mine[:4]}…"
    )
    assert mine[-1] == 0 and mine == sorted(mine, reverse=True), "Descending, ending at 0."
assert inference_timesteps(100, 20)[:3] == [95, 90, 85]
for bad in (0, 101):
    try:
        inference_timesteps(100, bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"{bad} steps must raise ValueError.")
print("Exercise 14.3 passed")

# More behaviour checks: lesson14_checks.py, next to this notebook.
from lesson14_checks import check_inference_timesteps

check_inference_timesteps(inference_timesteps)

## Exercise 14.4 — The reverse loop with explicit noise

**Purpose.** Run the process from noise to actions. All randomness is passed in, so the same inputs always give the same actions.

**Contract (shapes, units, masks).**

- `sample_actions(denoise, initial_noise, schedule, num_inference_steps, step_noises, clip_range=1.0)`.
- `denoise(x_t, t_batch) -> epsilon` where `t_batch` is a **`long` tensor `(B,)` filled with the current timestep** (same device as `x_t`). `initial_noise (B, H, A)`. `schedule` has `.alpha_bar` (the `(T,)` cumulative alphas) and `.num_train_timesteps` ($T$).
- `step_noises` is a sequence of `(B, H, A)` tensors: `step_noises[i]` is the noise for step `i`. There must be exactly `num_inference_steps - 1` of them (the final step uses none); otherwise raise `ValueError`.
- Use `inference_timesteps` (Exercise 14.3). Start from `x = initial_noise`. For each timestep: call `denoise(x, t_batch)`, then `x = ddpm_reverse_step(...)` (Exercise 14.2). **Carry the new `x` into the next step.** The previous timestep of the last entry is `-1`.
- Return the final `x`. Do not call a scheduler and do not draw random numbers.

**Method and source.** DDPM Algorithm 2 with the shortened timesteps of Exercise 14.3. The pinned `conditional_sample` is the same loop with `noise_scheduler.step` inside.

**Pointers.** Compute the list of timesteps once. Loop with `enumerate` so that you know the next timestep. Build `t_batch` with `torch.full((batch,), t, dtype=torch.long, device=x.device)`.

<details><summary>Detailed hint</summary>

```python
timesteps = inference_timesteps(schedule.num_train_timesteps, num_inference_steps)
if len(step_noises) != len(timesteps) - 1:
    raise ValueError("need one noise tensor for every step except the last")
x = initial_noise
for index, t in enumerate(timesteps):
    prev_t = timesteps[index + 1] if index + 1 < len(timesteps) else -1
    t_batch = torch.full((x.shape[0],), t, dtype=torch.long, device=x.device)
    epsilon = denoise(x, t_batch)
    noise = step_noises[index] if prev_t >= 0 else None
    x = ddpm_reverse_step(x, epsilon, t, prev_t, schedule.alpha_bar, noise, clip_range)
return x
```

</details>

More behaviour checks for this exercise: `check_sample_actions` in [lesson14_checks.py](lesson14_checks.py).

In [ ]:
# Implement: Exercise 14.4
def sample_actions(
    denoise: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    initial_noise: torch.Tensor,
    schedule: DDPMSchedule,
    num_inference_steps: int,
    step_noises: Sequence[torch.Tensor],
    clip_range: float | None = 1.0,
) -> torch.Tensor:
    # For each timestep: predict epsilon with denoise, take a reverse step, carry x to the next step.
    raise NotImplementedError("Complete Exercise 14.4: sample_actions")

In [ ]:
# Check: Exercise 14.4
schedule = DDPMSchedule.cosine(100)
torch.manual_seed(8)
net = nn.Linear(7, 7)


def toy(x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    return net(x) * (t.float().reshape(-1, 1, 1) / 100.0)


# (1) Same noise as the diffusers loop gives the same actions.
for steps in (3, 20, 100):
    scheduler = DDPMScheduler(
        num_train_timesteps=100,
        beta_schedule="squaredcos_cap_v2",
        clip_sample=True,
        clip_sample_range=1.0,
    )
    scheduler.set_timesteps(steps)
    g = torch.Generator().manual_seed(1)
    initial = torch.randn(2, 16, 7, generator=g)
    expected = initial
    for t in scheduler.timesteps:  # oracle loop (uses scheduler.step)
        expected = scheduler.step(
            toy(expected, torch.full((2,), int(t))), int(t), expected, generator=g
        ).prev_sample
    g2 = torch.Generator().manual_seed(1)
    initial2 = torch.randn(2, 16, 7, generator=g2)
    noises = make_step_noises(steps, initial2.shape, g2)
    close(
        sample_actions(toy, initial2, schedule, steps, noises, 1.0),
        expected,
        f"{steps} steps differ from the diffusers loop",
        atol=1e-4,
    )

# (2) Behaviour: a perfect denoiser recovers the clean window exactly, whatever the noise.
clean = torch.rand(2, 16, 7) * 1.6 - 0.8


def oracle(x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    ab = schedule.alpha_bar[t].reshape(-1, 1, 1)
    return (x - ab.sqrt() * clean) / (1 - ab).sqrt()


seen = []


def spy(x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    seen.append((x.clone(), t.clone()))
    return oracle(x, t)


start = torch.randn(2, 16, 7, generator=torch.Generator().manual_seed(2))
out = sample_actions(
    spy, start, schedule, 10, make_step_noises(10, start.shape, torch.Generator().manual_seed(3))
)
close(out, clean, "a perfect denoiser must recover the clean window", atol=1e-4)
assert [int(t[0]) for _, t in seen] == [90, 80, 70, 60, 50, 40, 30, 20, 10, 0], (
    "Visit inference_timesteps(100, 10) in order."
)
assert all(t.dtype == torch.long and t.shape == (2,) for _, t in seen), (
    "t_batch must be a long tensor of shape (B,)."
)
assert torch.equal(seen[0][0], start) and not torch.equal(seen[1][0], start), (
    "Carry each new sample into the next step."
)
# (3) Repeatable and strict about the noise count.
noises = make_step_noises(10, start.shape, torch.Generator().manual_seed(3))
assert torch.equal(
    sample_actions(toy, start, schedule, 10, noises),
    sample_actions(toy, start, schedule, 10, noises),
)
try:
    sample_actions(toy, start, schedule, 10, noises[:-1])
except ValueError:
    pass
else:
    raise AssertionError("A wrong number of step noises must raise ValueError.")
print("Exercise 14.4 passed")

# More behaviour checks: lesson14_checks.py, next to this notebook.
from lesson14_checks import check_sample_actions

check_sample_actions(sample_actions)

## View the sampler schedule

The slider changes only the selected timesteps. It does not run a denoiser.

In [ ]:
def show_sampling_steps(steps: int = 20) -> None:
    alpha = cosine_betas(100).cumprod(0)
    times = inference_timesteps(100, steps)
    plt.plot(alpha.numpy(), label="remaining signal")
    plt.scatter(times, alpha[times].numpy(), color="red", label="visited steps")
    plt.xlabel("training timestep")
    plt.ylabel("alpha bar")
    plt.title(f"{steps} denoising updates")
    plt.legend()
    plt.show()


display(
    widgets.interactive(
        show_sampling_steps,
        steps=widgets.IntSlider(value=20, min=2, max=100, step=2, description="updates"),
    )
)

## Full experiment — Run the learner sampler in LIBERO

The selected experiment uses a prepared policy, this notebook's sampler, the standard processors, and one bounded development-state rollout.

In [ ]:
CHECKPOINT = WORKSPACE / ".local-data/runs/diffusion-bowl-plate-reference-2000/checkpoint-002000"
POLICY, PREPROCESS, POSTPROCESS = load_prepared_diffusion(
    CHECKPOINT, device="cuda" if torch.cuda.is_available() else "cpu"
)
RUNNER = CallbackDiffusionPolicy(
    POLICY, sampler=sample_actions, slicer=executed_action_slice, num_inference_steps=5, seed=7
)

In [ ]:
ENV = open_libero_task("bowl_plate", 10, WORKSPACE / "configs", WORKSPACE / ".local-data")
try:
    RESULT = run_policy_episode(
        ENV,
        RUNNER,
        PREPROCESS,
        POSTPROCESS,
        "bowl_plate",
        10,
        WORKSPACE / "configs",
        max_steps=20,
        reset_seed=7,
    )
finally:
    ENV.close()
assert RESULT.steps <= 20 and RESULT.control_hz == 20
assert RUNNER.chunks_made >= 1
print({"steps": RESULT.steps, "success": RESULT.success, "sampler_calls": RUNNER.chunks_made})